In [ ]:
import pandas as pd
from scipy.stats import zscore

try:
    # Load the dataset
    df = pd.read_csv('./data/togodata.csv')
    print("Dataset loaded successfully.")

    # Check for missing values
    print("Missing values per column:\n", df.isna().sum())

    # Handle missing values (fill with mean for numeric columns)
    df.fillna(df.mean(numeric_only=True), inplace=True)
    print("Missing values handled by mean imputation.")

    # Check if 'GHI' column exists
    if 'GHI' not in df.columns:
        raise KeyError("Missing required column: 'GHI'")

    # Identify outliers using Z-score
    df['z_ghi'] = zscore(df['GHI'])

    # Filter outliers
    df_outliers = df[df['z_ghi'].abs() > 3]
    print("Outliers in GHI:\n", df_outliers)

except FileNotFoundError:
    print("Error: File not found. Please check the file path.")
except pd.errors.EmptyDataError:
    print("Error: The CSV file is empty.")
except KeyError as e:
    print(f"Column error: {e}")
except ValueError as e:
    print(f"Value error during processing: {e}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from windrose import WindroseAxes
import pandas as pd

# Assuming df is already defined and loaded

# Line plots
try:
    df[['GHI', 'DNI', 'DHI', 'Tamb']].plot(figsize=(10, 6))
    plt.title("Line plots for GHI, DNI, DHI, Tamb")
    plt.show()
except KeyError as e:
    print(f"Line plot error: Missing column {e}")
except Exception as e:
    print(f"Unexpected error in line plot: {e}")

# Correlation heatmap
try:
    plt.figure(figsize=(8, 6))
    sns.heatmap(df.corr(numeric_only=True), annot=True, cmap="coolwarm", fmt='.2f')
    plt.title('Correlation Heatmap')
    plt.show()
except Exception as e:
    print(f"Error in correlation heatmap: {e}")

# Scatter plot
try:
    sns.scatterplot(x='RH', y='GHI', data=df)
    plt.title('RH vs GHI')
    plt.show()
except KeyError as e:
    print(f"Scatter plot error: Missing column {e}")
except Exception as e:
    print(f"Unexpected error in scatter plot: {e}")

# Histogram
try:
    df['GHI'].hist(bins=30, color='skyblue', edgecolor='black')
    plt.title('Histogram of GHI')
    plt.show()
except KeyError:
    print("Histogram error: Column 'GHI' not found.")
except Exception as e:
    print(f"Unexpected error in histogram: {e}")


try:
    if 'Wind Direction' in df.columns and 'Wind Speed' in df.columns:
        fig = plt.figure(figsize=(6, 6))
        ax = fig.add_subplot(111, projection='windrose')
        ax.bar(df['Wind Direction'], df['Wind Speed'], bins=8, opening=0.8, edgecolor='white')
        ax.set_legend()
        plt.title('Wind Rose')
        plt.show()
    else:
        raise KeyError("Columns 'Wind Direction' and/or 'Wind Speed' not found.")
except KeyError as e:
    print(f"Wind rose plot error: {e}")
except Exception as e:
    print(f"Unexpected error in wind rose plot: {e}")

# Bubble chart
try:
    plt.scatter(df['GHI'], df['Tamb'], s=df['RH']*10, alpha=0.5)
    plt.xlabel('GHI')
    plt.ylabel('Tamb')
    plt.title('Bubble chart: GHI vs Tamb')
    plt.show()
except KeyError as e:
    print(f"Bubble chart error: Missing column {e}")
except Exception as e:
    print(f"Unexpected error in bubble chart: {e}")


In [ ]:
from sklearn.preprocessing import StandardScaler
import pandas as pd

try:
    # Define the columns to scale
    columns_to_scale = ['GHI', 'DNI', 'DHI', 'Tamb']
    
    # Check if all required columns exist
    missing_cols = [col for col in columns_to_scale if col not in df.columns]
    if missing_cols:
        raise KeyError(f"Missing columns for scaling: {missing_cols}")
    
    # Scale the features
    scaler = StandardScaler()
    df[columns_to_scale] = scaler.fit_transform(df[columns_to_scale])
    print("Feature scaling completed successfully.")

    # Create new feature: GHI_DHI_ratio
    if 'GHI' in df.columns and 'DHI' in df.columns:
        df['GHI_DHI_ratio'] = df['GHI'] / (df['DHI'] + 1e-5)  # Avoid division by zero
        print("New feature 'GHI_DHI_ratio' created successfully.")
    else:
        raise KeyError("Missing columns 'GHI' or 'DHI' for feature creation.")

    # Preview the updated dataframe
    print(df.head())

except KeyError as e:
    print(f"Key error: {e}")
except ValueError as e:
    print(f"Value error: {e} — make sure the columns are numeric.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")
